Linear Regression Intuition

The main goal of this document is to provide a straightforward mathematical intuition into how the linear regression models works behind the scenes, from the OLS(Ordinary Least Squares) method, the GD(Gradient Descent) optimization algorithm to inference.

Contents:
1. Introduction
2. Least Squares Method
3. Gradient Descent & Variations
4. Regularization
4. Use cases

Introduction

Linear regression is a supervised learning algorithm that predicts a numerical(continuous) variable y based on a training set X. Although it can be used for multiple training features, for simplicity purposes this article will use univariate linear regression.

Equation: f : R^m -> R, f(x) = w1 * x1 + w2 * x2 + ... + wm*xm + b (m >= 2)

We will be using f:R^m -> IR, f(x) = w * x + b

Say we want to predict a house's price based on square footage alone(this is practically impossible since there are a billion other factors that dictate a house's price such as location number of rooms/bedrooms/bathrooms, views etc. but the relationship is still mostly linear), we want to teach the algorithm(or ourselves as well) the best values for w and b.

Quick explanation:
    w - the slope of the square footage(how much the price grows per square foot/a single unit of the feature, as we assume the relationship is linear this works just fine)
    b - tbe bias (the starting point of the y variable, houses usually cost more than 0$ so we need to start from a certain minimum)

Now, we will dive into how we would want to solve this problem that we have framed using calculus (and linear algebra)

To do: de facut frumos si explicat acest experiment. plan: mai intai folosesc valori mari si coeficienti ficsi fara gaussian noise ca sa arat de ce avem nevoie de scalarea datelor. sa arat cat de greu converge GD-ul.

In [ ]:
import numpy as np
X = np.array([1500, 2000, 4000, 1230, 1400, 3500, 2300, 2500, 1700, 2900, 6000, 5000, 4500, 1200])
len(X)
y = 15 * X + 100000

14

mai intai facem LS ca e mai usor si straightforward si usor de inteles si dupa facem GD.

OLS(Ordinary Least Squares)

OLS is the most straightforward method to find the best parameters(weights and bias) for your regression problem, what it does in a nutshell is it sets the derivative of the MSE function(Mean Squared Error) to 0 as it is strictly convex and the only point where the derivative is 0 is the global mininum.

errors = [(w * x + b - y_i) for x, y_i in X, y]
MSE = mean(w * x + b) = 1/n * sum(square(errors))

Why use MSE? Let's look at a single error: (y_predicted - y_true) ** 2. We square it because for most regression problems, we see larger errors as much more of a concern than little ones, example: predicting a house as being worth 400k instead of 500k will have mean an error of 10 million(massive), whereas predicting a house as being worth 250k instead of 280k means an error of 900k, basically ten times less for an error thats just three times smaller.

After we calculate the derivative with respect to w: MSE(w)' = 2 / n * sum(i = 1,2, ..., n) ((w * x_i + b - y_i) * x_i)

Since we only calculated the derivative to set it to 0 and it doesn't actually affect the weights iteratively, the i-th feature's size (compared to others) doesn't affect us.


Setting the derivative to 0 and using vector forms: 2/n (w * X - y)*X.T = 0 -> w * X * X.T = y * X.T -> w = y * X.t... we get the optimal values for the coefficients(the ones with the smallest residuals for the house prices)

Despite its simplicity and effectiveness(it gives us EXACTLY the optimal coefficients), the OLS method is not always the one we want to use. For large datasets, the time complexity of the matrix multiplication operation used for computing the coefficients is O(n^3). Also, we cannot rely on the MSE being strictly convex, therefore sometimes simply setting the derivative to 0 is not enough, as the derivative is not guaranteed to be 0-injective anymore. The widely used alternative is gradient descent.

Gradient Descent 

Before we dive into Gradient Descent, I want to offer a quick explanation on why subtracting the gradient of a function at a given point decreases it towards a minimum. In order to do that, we need to dive a bit into vector calculus: the gradient of a function f(x) where x is an n dimensional vector x = [x1, x2, x3, ..., xn] is the vector of the partial derivatives of f with respect to x_i where i ranges from 1 to n. Now how do we calculate the fastest way to get to the minimum? We get the fastest descent. What is the fastest descent? The opposite of the fastest way uphill, which is exactly the gradient, but don't take my word for it, here's mathematical proof:

### Directional Derivative

For the unit direction vector $\mathbf{d}$, define

$$
g(h) = f(\mathbf{x} + h\mathbf{d}).
$$

Then the directional derivative of $f$ at $\mathbf{x}$ in the direction $\mathbf{d}$ is

$$
\begin{aligned}
D_{\mathbf{d}}f(\mathbf{x})
&= \lim_{h \to 0} \frac{f(\mathbf{x} + h\mathbf{d}) - f(\mathbf{x})}{h} \\
&= \lim_{h \to 0} \frac{g(h) - g(0)}{h} \\
&= g'(0) \\
&= \nabla f(\mathbf{x}) \cdot \mathbf{d}.
\end{aligned}
$$

Now that we understand why gradient descent is the way that points directly downhill, let's get into it with a real problem.

Let

$$
X =
\begin{pmatrix}
 x_{11} & x_{12} & \cdots & x_{1n} \\
 x_{21} & x_{22} & \cdots & x_{2n} \\
 \vdots & \vdots & \ddots & \vdots \\
 x_{m1} & x_{m2} & \cdots & x_{mn}
\end{pmatrix}
\in \mathbb{R}^{m \times n},
\qquad
\mathbf{y} =
\begin{pmatrix}
 y_1 \\ y_2 \\ \vdots \\ y_m
\end{pmatrix}
\in \mathbb{R}^{m \times 1}.
$$

Here, $X$ contains $m$ training examples and $n$ features. The parameter vector is

$$
\mathbf{w} =
\begin{pmatrix}
 w_1 \\ w_2 \\ \vdots \\ w_n
\end{pmatrix}
\in \mathbb{R}^{n \times 1},
$$

and $\mathbf{x}_i \in \mathbb{R}^{n \times 1}$ denotes the $i$-th row of $X$, written as a column vector. For a given example, the prediction is

$$
\hat{y}_i = \mathbf{x}_i^{\mathsf{T}}\mathbf{w} + b,
$$

where $b$ is the bias term.

### The loss function: mean squared error

The mean squared error measures the average squared difference between the predictions and the true labels:

$$
J(\mathbf{w}, b)
= \frac{1}{m}\sum_{i=1}^{m}
\left(\mathbf{x}_i^{\mathsf{T}}\mathbf{w} + b - y_i\right)^2.
$$

The squared error penalizes large mistakes more heavily, while keeping the loss differentiable and therefore suitable for optimization.

### Computing the gradient

To differentiate with respect to a single weight $w_j$, apply the chain rule. The derivative of the squared error contributes a factor of $2$, and the derivative of the prediction with respect to $w_j$ is $x_{ij}$:

$$
\frac{\partial J}{\partial w_j}
= \frac{2}{m}\sum_{i=1}^{m}
\left(\mathbf{x}_i^{\mathsf{T}}\mathbf{w} + b - y_i\right)x_{ij}.
$$

Stacking these partial derivatives for $j=1,\ldots,n$ gives the gradient with respect to the entire weight vector:

$$
\nabla_{\mathbf{w}} J(\mathbf{w}, b)
= \frac{2}{m}\sum_{i=1}^{m}
\left(\mathbf{x}_i^{\mathsf{T}}\mathbf{w} + b - y_i\right)\mathbf{x}_i.
$$

In matrix form, with $\mathbf{e} = X\mathbf{w} + b\mathbf{1} - \mathbf{y}$ as the vector of residuals,

$$
\nabla_{\mathbf{w}} J(\mathbf{w}, b)
= \frac{2}{m}X^{\mathsf{T}}\mathbf{e}.
$$

### The gradient-descent update

The gradient points in the direction of steepest ascent. Therefore, moving in the opposite direction decreases the loss locally. With learning rate $\eta > 0$, the weights are updated by

$$
\boxed{
\mathbf{w}
\leftarrow
\mathbf{w} - \eta\,\nabla_{\mathbf{w}}J(\mathbf{w}, b)
}
$$

or, equivalently,

$$
\mathbf{w}
\leftarrow
\mathbf{w}
- \eta\,\frac{2}{m}X^{\mathsf{T}}
\left(X\mathbf{w} + b\mathbf{1} - \mathbf{y}\right).
$$

If the bias is optimized as well, its gradient is

$$
\frac{\partial J}{\partial b}
= \frac{2}{m}\sum_{i=1}^{m}
\left(\mathbf{x}_i^{\mathsf{T}}\mathbf{w} + b - y_i\right),
$$

and its update is

$$
 b \leftarrow b - \eta\,\frac{\partial J}{\partial b}.
$$

Repeating these updates moves $(\mathbf{w}, b)$ toward parameters that minimize the loss. The optimization objective is

$$
\boxed{
(\mathbf{w}^*, b^*)
= \underset{\mathbf{w},b}{\operatorname{arg\,min}}\;J(\mathbf{w}, b)
}.
$$

Now you understand Gradient Descent :)

### Stochastic Gradient Descent

When we have a large training set, using the gradient descent algorithm can be slow, because it goes through all m rows(examples), every iteration. To fix that we use SGD, which uses only a random row of the data when training each iteration.

### Mini Batch Gradient Descent

SGD doesn't achieve as optimal a result as plain GD, so, as is in life, the answer is somewhere in the middle: mini-batch. Instead of using the whole training set each iteration (the whole batch) or using a single row we use a random subset of fixed size (mini-batch) because it keeps the optimality closer to plain GD, while keeping the process faster.

## Regularization

Plain Linear Regression tends to overfit(memorize the training data) instead of generalizing, in order to fix that we add a penalty(regularization term) to the loss function (MSE). The cause of the overfitting are the large coefficients, which tend to shift very much in the direction of every single row or outlier and therefore are not very rigid.